# 03 - Additional Dataset Experiments

This notebook is reserved for experiments on datasets that are outside the paper scope.

Use this notebook for extension runs only. Official reproduction tables should stay in notebook 01 and notebook 02.


## Dataset Choice

Chosen dataset: `beans`

Why this dataset is a good local test target:
- It is outside the paper scope, so it satisfies the project requirement for a new dataset.
- It is a small RGB classification benchmark with only 3 classes, which makes it realistic to run locally on CPU.
- It can reuse the ImageNet-style encoder path already present in the repo.
- Its low class count is a useful stress test for the plug-and-play module because over-splitting becomes easy to observe.


In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import sys

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULT_DIR = ROOT / 'data' / 'scan_results' / 'additional_datasets'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_CSV = RESULT_DIR / 'beans_comparison.csv'

RUN_EXPERIMENT = False
if RUN_EXPERIMENT or not RESULT_CSV.exists():
    subprocess.run(['python', 'scripts/generate_extension_tables.py'], check=True, cwd=ROOT)

history = pd.read_csv(RESULT_CSV)
display(history)


,Dataset,Method,K0,K*,ACC(%),NMI(%),ARI(%),Notes
0,beans,SCAN (local),3,3,61.71875,22.560666,22.045674,Small RGB benchmark outside the paper scope wi...
1,beans,Ours (local),5,4,50.78125,22.699824,19.247794,Small RGB benchmark outside the paper scope wi...


## Recommended Local Configuration

The saved CSV uses the following two runs:
- `SCAN (local)` with fixed `K=3`
- `Ours (local)` with `K0=5` to test whether the module can contract toward the true class count

This pairing keeps the extension compact while still testing both the baseline branch and the unknown-`K` branch.


## Analysis

Report-ready interpretation:
- The local `SCAN` baseline reaches `K*=3` on `beans`, which is the correct class count, and gives the stronger clustering score in this extension.
- The local `Ours` run starts from `K0=5` and finishes at `K*=4`, so it does reduce the over-estimated cluster count but does not fully converge to the true `K=3`.
- This suggests the current plug-and-play thresholds are tuned more comfortably for the 10-class and 20-class benchmarks used in the paper than for a tiny 3-class dataset.
- The result is still useful for the report because it shows the method can generalize beyond the paper domain, but its split and merge control remains sensitive when the target cluster count is very small.
